In [1]:
from pathlib import Path
import shutil

res_dir = Path("results/data")

# 1) Ergebnis-CSVs löschen
for fn in ("raw_results.csv","best_results.csv"):
    p = res_dir/fn
    if p.exists(): 
        p.unlink()

# 2) Joblib-Cache löschen
cache_dir = res_dir/"pipeline_cache"
if cache_dir.exists():
    shutil.rmtree(cache_dir)
print("Cache und alte Ergebnisse gelöscht.")


Cache und alte Ergebnisse gelöscht.


In [ ]:
# %% [code]
from pathlib import Path
import pandas as pd
import numpy as np
import ast
from tqdm.auto import tqdm

from FIT_python.pipeline.pipeline_wrapper_sex import PipelineWrapper
from FIT_python.config import RESULTS_DATA_DIR

# 1) Einmaliges Setup: Clean → Split → Summary
prep = PipelineWrapper(
    model_keys=["log_reg"],     # Dummy
    impute_method=None,
    outlier_method=None,
    scaler_method=None,
    reduce_pre_method=None,
    reduce_post_method=None,
    fs_method=None,
    fs_k=None,
    validation_strategy="custom"
)
prep.prepare()


# 2) Output-Dateien definieren
res_dir = Path(RESULTS_DATA_DIR)
raw_csv  = res_dir / "raw_results.csv"
best_csv = res_dir / "best_results.csv"

# Alte Dateien entfernen (optional)
if raw_csv.exists():  raw_csv.unlink()
if best_csv.exists(): best_csv.unlink()

best_score = -np.inf
best_params = None


# 3) Parameterraster und Sampling
from sklearn.model_selection import ParameterSampler

param_dist = {
    "impute_method":      [None],
    "outlier_method":     [None, "zscore"],
    "scaler_method":      [None, "standard"],
    "reduce_pre_method":  [None, "pca", "umap"],
    "reduce_post_method": [None, "pca", "umap"],
    "fs_method":          [None, "forward", "random_forest"],
    "fs_k":               [0, 2, 4, 5, 10, 100],
    "model_key":          ["log_reg","rf","et","knn","svm","lda","xgb"],
}
# nur 100 zufällige Kombinationen statt aller ~5 000
grid = list(ParameterSampler(param_dist, n_iter=100, random_state=42))
print(f"Anzahl gezogener Konfigurationen: {len(grid)}")


# Hilfsfunktion für NaN → None
def clean_none(x):
    return None if pd.isna(x) else x


# 4) Serieller Loop mit kontinuierlichem CSV-Update
for params in tqdm(grid, desc="Random-Search"):
    # a) Parameter auf Python-Typ konvertieren
    impute   = clean_none(params["impute_method"])
    outlier  = clean_none(params["outlier_method"])
    scaler   = clean_none(params["scaler_method"])
    red_pre  = clean_none(params["reduce_pre_method"])
    red_post = clean_none(params["reduce_post_method"])
    fs_meth  = clean_none(params["fs_method"])
    fs_k_raw = params["fs_k"]
    fs_k     = None if (pd.isna(fs_k_raw) or fs_k_raw == 0) else int(fs_k_raw)
    model    = clean_none(params["model_key"])

    # b) Pipeline trainieren
    pw = PipelineWrapper(
        model_keys=[model],
        impute_method=     impute,
        outlier_method=    outlier,
        scaler_method=     scaler,
        reduce_pre_method= red_pre,
        reduce_post_method=red_post,
        fs_method=         fs_meth,
        fs_k=              fs_k,
        validation_strategy="custom"
    )
    df_raw = pw.train()

    # c) classification_report evtl. von String in Dict konvertieren
    df_raw["classification_report"] = df_raw["classification_report"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

    # d) Balanced Accuracy (macro avg recall) berechnen
    df_raw["balanced_accuracy"] = df_raw["classification_report"].apply(
        lambda cr: cr["macro avg"]["recall"]
    )

    # e) Metadaten anhängen
    for k, v in params.items():
        df_raw[k] = v

    # f) Raw-Ergebnisse anhängen
    df_raw.to_csv(raw_csv, mode="a", header=not raw_csv.exists(), index=False)

    # g) globalen Best-Score updaten
    mean_score = df_raw["balanced_accuracy"].mean()
    if mean_score > best_score:
        best_score  = mean_score
        best_params = {**params, "best_score": best_score}
        # sofort neue Best-CSV schreiben (überschreibt vorherige)
        pd.DataFrame([best_params]).to_csv(best_csv, index=False)


# 5) Am Ende haben wir:
#  - raw_results.csv  ← alle Runs aller Spezies & Parameter
#  - best_results.csv ← immer die beste Kombi mit Score
print("Fertig! Siehe:")
print(f"  • {raw_csv}")
print(f"  • {best_csv}")



📥 Schritt 1: Datenimport & Cleaning
[REPORT] cleaned Aj_Someringii: shape=(370, 141)
[REPORT] cleaned Amur_Tiger: shape=(605, 133)
[REPORT] cleaned Bengal_tiger: shape=(586, 133)
[REPORT] cleaned cheetah: shape=(781, 141)
[REPORT] cleaned Eurasian_Otter_(1): shape=(628, 214)
[REPORT] cleaned Giant_Panda: shape=(523, 129)
[REPORT] cleaned Jubatus: shape=(566, 141)
[REPORT] cleaned Lowlandtapir: shape=(426, 126)
[REPORT] cleaned Mountain_Lion: shape=(535, 138)
[REPORT] cleaned White_Rhino: shape=(1273, 117)
[SUCCESS] clean_all completed.

✂️ Schritt 2: Splitting & Fold-Zuordnung
✅ Fold-Methode verwendet: stratified_group
Fold
0     99
1    110
2     91

📊 aj_someringii – train – 300 Zeilen | 20 Individuen | 43 Trails
sex
f    175
m    125

📊 aj_someringii – test – 70 Zeilen | 5 Individuen | 10 Trails
sex
m    56
f    14

📊 aj_someringii – inference – 0 Zeilen | 0 Individuen | 0 Trails
Series([], )

✅ aj_someringii gespeichert (csv & parquet)
✅ Fold-Methode verwendet: stratified_group
Fo

C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\src\FIT_python\pipeline\split_utils.py:162: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["Fold"] = fold_ids
C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\src\FIT_python\pipeline\split_utils.py:162: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["Fold"] = fold_ids



📊 eurasian_otter_(1) – test – 184 Zeilen | 14 Individuen | 25 Trails
sex
f    112
m     72

📊 eurasian_otter_(1) – inference – 81 Zeilen | 8 Individuen | 8 Trails
sex
unknown    81

✅ eurasian_otter_(1) gespeichert (csv & parquet)
✅ Fold-Methode verwendet: stratified_group
Fold
0    135
1    149
2    121

📊 giant_panda – train – 405 Zeilen | 33 Individuen | 59 Trails
sex
f    222
m    183

📊 giant_panda – test – 118 Zeilen | 9 Individuen | 18 Trails
sex
f    79
m    39

📊 giant_panda – inference – 0 Zeilen | 0 Individuen | 0 Trails
Series([], )

✅ giant_panda gespeichert (csv & parquet)
✅ Fold-Methode verwendet: stratified_group
Fold
0    161
1    153
2    140

📊 jubatus – train – 454 Zeilen | 32 Individuen | 65 Trails
sex
f    272
m    182

📊 jubatus – test – 112 Zeilen | 8 Individuen | 16 Trails
sex
m    56
f    56

📊 jubatus – inference – 0 Zeilen | 0 Individuen | 0 Trails
Series([], )

✅ jubatus gespeichert (csv & parquet)
✅ Fold-Methode verwendet: kfold
Fold
0    106
1    105
2  

C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\src\FIT_python\pipeline\split_utils.py:162: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["Fold"] = fold_ids
C:\Users\Frede\OneDrive\Projects and Disschaptors\FIT_package\src\FIT_python\pipeline\split_utils.py:162: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df["Fold"] = fold_ids



📊 eurasian_otter_(1) – train – 363 Zeilen | 27 Individuen | 48 Trails
sex
f    194
m    169

📊 eurasian_otter_(1) – test – 184 Zeilen | 14 Individuen | 25 Trails
sex
f    112
m     72

📊 eurasian_otter_(1) – inference – 81 Zeilen | 8 Individuen | 8 Trails
sex
unknown    81

✅ eurasian_otter_(1) gespeichert (csv & parquet)
✅ Fold-Methode verwendet: stratified_group
Fold
0    135
1    149
2    121

📊 giant_panda – train – 405 Zeilen | 33 Individuen | 59 Trails
sex
f    222
m    183

📊 giant_panda – test – 118 Zeilen | 9 Individuen | 18 Trails
sex
f    79
m    39

📊 giant_panda – inference – 0 Zeilen | 0 Individuen | 0 Trails
Series([], )

✅ giant_panda gespeichert (csv & parquet)
✅ Fold-Methode verwendet: stratified_group
Fold
0    161
1    153
2    140

📊 jubatus – train – 454 Zeilen | 32 Individuen | 65 Trails
sex
f    272
m    182

📊 jubatus – test – 112 Zeilen | 8 Individuen | 16 Trails
sex
m    56
f    56

📊 jubatus – inference – 0 Zeilen | 0 Individuen | 0 Trails
Series([], )

✅ j

Random-Search:   0%|          | 0/100 [00:00<?, ?it/s]

c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\Frede\anaconda3\envs\FIT_python_conda_env\Lib\site-packages\sklearn\utils\deprecation.py:132: F

In [ ]:
import multiprocessing

print(f"CPU-Kerne verfügbar: {multiprocessing.cpu_count()}")
